In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# =============================================================================
# Inputs:
#   1) shares: DataFrame, index = date (DatetimeIndex), columns = series_id
#   2) emp:    DataFrame, index = date (DatetimeIndex), columns = series_id (employment levels)
#   3) mapping: CSV with columns at least ["series_id","parent_series_id"]
#
# =============================================================================

shares = pd.read_csv("b1a_employment_shares.csv", index_col=0, parse_dates=True)
emp    = pd.read_parquet("b1a_wide_seriesid.parquet")

MAPPING_PATH = "b1a_mapping_with_parent.csv"  # or "/mnt/data/b1a_mapping.csv"

mapping = pd.read_csv(MAPPING_PATH, dtype=str)
mapping.columns = mapping.columns.str.strip()
if "parent_series_id" not in mapping.columns:
    raise KeyError("mapping must contain 'parent_series_id' (your denominator series_id).")

# Keep only series that exist in shares / emp
mapping = mapping.dropna(subset=["series_id", "parent_series_id"]).copy()


# =============================================================================
# 1) Trailing 60-Month MA Deviation (percent deviation from trailing mean)
# ε_t = (s_t - MA60_{t-1}) / MA60_{t-1}
# where MA60_{t-1} is the mean of the previous 60 months (excludes current).
# =============================================================================
def detrend_trailing_ma60(shares: pd.DataFrame, window=60) -> pd.DataFrame:
    ma = shares.rolling(window=window, min_periods=window).mean().shift(1)
    eps = (shares - ma) / ma
    return eps

detrended_ma60 = detrend_trailing_ma60(shares, window=60)


# =============================================================================
# 2) Hamilton Filter on Log Share (lags 24–27 months, full-sample regression)
# log(s_t) = b0 + b1 log(s_{t-24}) + ... + b4 log(s_{t-27}) + e_t
# residual e_t returned (aligned to t where all lags exist).
# =============================================================================
def detrend_hamilton_logshare(shares: pd.DataFrame, lags=(24, 25, 26, 27), eps=1e-12) -> pd.DataFrame:
    out = pd.DataFrame(index=shares.index, columns=shares.columns, dtype=float)

    for col in shares.columns:
        s = shares[col].astype(float)

        # log-share; handle zeros/negatives by setting them to NaN (shares should be >=0)
        s = s.where(s > 0)
        y = np.log(s)

        X = pd.concat([y.shift(L) for L in lags], axis=1)
        X.columns = [f"lag{L}" for L in lags]

        df = pd.concat([y, X], axis=1).dropna()
        if df.empty:
            continue

        y_reg = df.iloc[:, 0]
        X_reg = sm.add_constant(df.iloc[:, 1:], has_constant="add")

        res = sm.OLS(y_reg, X_reg).fit()
        out.loc[df.index, col] = res.resid

    return out

detrended_hamilton = detrend_hamilton_logshare(shares, lags=(24, 25, 26, 27))


# =============================================================================
# 3) Log Polynomial Decomposition (N and D separately; cubic trend)
# For EACH series_id and EACH denominator series:
#   log(E_t) = poly3(t) + resid_t
# Then detrended share residual:
#   eps^S_t = resid_i,t - resid_parent,t
# This gives an exact decomposition into numerator vs denominator deviations.
# =============================================================================
def poly3_residuals_log_emp(emp: pd.DataFrame, degree=3) -> pd.DataFrame:
    """
    Fit log(emp_col) on [1, t, t^2, t^3] for each column independently (full sample),
    return residuals. Uses only non-missing, positive employment.
    """
    out = pd.DataFrame(index=emp.index, columns=emp.columns, dtype=float)

    # time index as 0..T-1
    t = np.arange(len(emp.index), dtype=float)
    X = np.column_stack([t**k for k in range(0, degree + 1)])  # includes constant
    # We'll subselect rows where y is non-missing per series; refit per series.

    for col in emp.columns:
        y = emp[col].astype(float)
        y = y.where(y > 0)
        y = np.log(y)

        mask = y.notna().values
        if mask.sum() < (degree + 2):  # need enough points
            continue

        X_i = X[mask, :]
        y_i = y.values[mask]

        res = sm.OLS(y_i, X_i).fit()
        # residuals aligned to masked rows
        out.loc[emp.index[mask], col] = res.resid

    return out

# Residuals of log employment for all series in emp
logemp_resid = poly3_residuals_log_emp(emp, degree=3)


def detrended_share_from_logemp_resids(mapping: pd.DataFrame,
                                      resid_logemp: pd.DataFrame) -> pd.DataFrame:
    """
    Build detrended share residuals eps^S = resid(series) - resid(parent_series)
    using mapping with parent_series_id.
    Columns are series_id (numerator series).
    """
    # keep rows where both numerator and parent exist in resid_logemp
    keep = mapping["series_id"].isin(resid_logemp.columns) & mapping["parent_series_id"].isin(resid_logemp.columns)
    m = mapping.loc[keep, ["series_id", "parent_series_id"]].drop_duplicates()

    out = pd.DataFrame(index=resid_logemp.index, dtype=float)

    for sid, pid in zip(m["series_id"], m["parent_series_id"]):
        out[sid] = resid_logemp[sid] - resid_logemp[pid]

    return out

detrended_poly3 = detrended_share_from_logemp_resids(mapping, logemp_resid)


# =============================================================================
# Save 
# =============================================================================
logemp_resid.to_csv("detrended/detrended_logemp.csv")
detrended_ma60.to_csv("detrended/detrended_share_ma60.csv")
detrended_hamilton.to_csv("detrended/detrended_share_hamilton_logshare.csv")
detrended_poly3.to_csv("detrended/detrended_share_poly3_logempdiff.csv")

print("Saved:")
print(" - /detrended/detrended_logemp.csv")
print(" - /detrended/detrended_share_ma60.csv")
print(" - /detrended/detrended_share_hamilton_logshare.csv")
print(" - /detrended/detrended_share_poly3_logempdiff.csv")

/var/folders/x3/1282rh0s7_b240x5mjx1zlmc0000gn/T/ipykernel_8225/471468523.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[sid] = resid_logemp[sid] - resid_logemp[pid]
/var/folders/x3/1282rh0s7_b240x5mjx1zlmc0000gn/T/ipykernel_8225/471468523.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[sid] = resid_logemp[sid] - resid_logemp[pid]
/var/folders/x3/1282rh0s7_b240x5mjx1zlmc0000gn/T/ipykernel_8225/471468523.py:129: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.

Saved:
 - /detrended/detrended_logemp.csv
 - /detrended/detrended_share_ma60.csv
 - /detrended/detrended_share_hamilton_logshare.csv
 - /detrended/detrended_share_poly3_logempdiff.csv
